# Read/write hardening: single-token, prefix, hard-decoy (Exp 1 + 6)
Protects the *fact-specific* claim. Per tokenizer we report the single-token fraction and the prefix-unique fraction, then re-run the dissociation on the single-token subset and on the prefix-unambiguous subset. We also build the conservative hard-decoy headline table.

In [ ]:
# --- environment (pins matching the pipeline) ---
# transformers==4.46.2  numpy==1.26.4  ; PyTorch nightly cu128 on newer instances.
import os, gc, json, math, pathlib
import numpy as np, torch
from tqdm.auto import tqdm
import rw_core as rc          # tested core (rw_core_smoketest.py: 19/19)
import rw_modelio as mio      # model IO / hooks / generation

ART = pathlib.Path(os.environ.get("RW_ART", "artifacts")); ART.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def save(obj, name):
    p = ART / name
    np.savez_compressed(p, **obj) if name.endswith(".npz") else \
        p.write_text(json.dumps(obj, indent=2, default=float))
    print("saved", p)

def exists(name):  # skip-if-exists guard
    return (ART / name).exists()


In [ ]:
MODELS = [
    "meta-llama/Llama-3.1-8B", "meta-llama/Llama-3.2-3B", "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B",
    "mistralai/Mistral-7B-v0.1",
]

In [ ]:
# ============================================================================
# DATA ADAPTER  --  wire this to your existing read/write cache.
# Return a list of item dicts (schema in rw_core docstring). The fields used
# downstream are listed per-cell. This is the ONLY place that knows your layout.
# ============================================================================
def load_items(model_name):
    """TODO: load your per-item records for `model_name`.
    Required keys vary by notebook; each cell asserts what it needs."""
    raise NotImplementedError("wire load_items() to your read/write cache")


In [ ]:
# Experiment 1: per-item single-token / prefix flags.
# Needs per item: gold_aliases, gold_first_tok, decoy_first_toks (+ read/write fields).
from transformers import AutoTokenizer
summary = {}
all_items = {}
for name in MODELS:
    if exists(f"flags_{name.split('/')[-1]}.json"):
        continue
    tok = AutoTokenizer.from_pretrained(name)
    items = load_items(name)
    for it in items:
        f = rc.single_token_flags(tok, it["gold_aliases"],
                                  it["decoy_first_toks"], it["gold_first_tok"])
        it.update(f)
    n = len(items)
    summary[name] = {
        "frac_single_token": np.mean([it["is_single_token"] for it in items]),
        "frac_prefix_unique": np.mean([it["prefix_unique"] for it in items]),
        "n": n,
    }
    all_items[name] = items
    save(summary[name], f"flags_{name.split('/')[-1]}.json")
print(json.dumps(summary, indent=2))

In [ ]:
# Experiment 1: dissociation on robustness subsets vs full set (hard decoy).
rows = []
for name, items in all_items.items():
    full = rc.readwrite_table(items, criterion="hard")
    st   = rc.subset_dissociation(items, "is_single_token", criterion="hard")
    pu   = rc.subset_dissociation(items, "prefix_unique",   criterion="hard")
    rows.append({"model": name,
                 "full_readable": full["readable_rate"], "full_nottop": full["not_top_rate"],
                 "single_readable": st["readable_rate"], "single_nottop": st["not_top_rate"],
                 "prefixuniq_readable": pu["readable_rate"], "prefixuniq_nottop": pu["not_top_rate"]})
save({"rows": rows}, "exp1_subset_dissociation.json")
for r in rows: print(r)

**Read this:** if `single_nottop` and `prefixuniq_nottop` stay high and close to `full_nottop`, the dissociation is not a prefix artifact and the fact-specific claim holds. If they drop sharply, you've located the boundary — report it in limitations rather than overclaiming.

In [ ]:
# Experiment 6: hard-decoy headline table (mean vs hard, with not-top under both).
head = []
for name, items in all_items.items():
    mean = rc.readwrite_table(items, criterion="mean")
    hard = rc.readwrite_table(items, criterion="hard")
    head.append({"model": name,
                 "readable_mean": mean["readable_rate"], "nottop_mean": mean["not_top_rate"],
                 "readable_hard": hard["readable_rate"], "nottop_hard": hard["not_top_rate"],
                 "n_fail": hard["n_fail"]})
save({"rows": head}, "exp6_hard_decoy_headline.json")
import pandas as pd
display(pd.DataFrame(head).round(3))

Promote the **hard-decoy** column to the headline; keep mean-decoy as the permissive comparison. The hard criterion (gold beats the strongest same-type decoy) is the one that answers the type-priming objection.